In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import re
from sklearn.metrics import accuracy_score, classification_report
from transformers import BertModel, BertTokenizerFast
import os

In [ ]:
# 定义数据集类
class EventRelationDataset(Dataset):
    def __init__(self, data_path, tokenizer, max_len=512):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.data = self.load_data(data_path)

    def load_data(self, data_path):
        """加载数据并处理五元组中的'无'值"""
        processed_data = []

        with open(data_path, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line.strip())

                # 提取关系类型作为标签
                relation_type = item['relation_type']

                # 提取事件信息
                first_event_text = item['first_event']['sentence_text']
                second_event_text = item['second_event']['sentence_text']

                # 处理五元组，将"无"替换为空字符串
                first_event_tuple = self.process_event_tuple(item['first_event']['event_tuple'])
                second_event_tuple = self.process_event_tuple(item['second_event']['event_tuple'])

                processed_data.append({
                    'relation_type': relation_type,
                    'first_event_text': first_event_text,
                    'first_event_tuple': first_event_tuple,
                    'second_event_text': second_event_text,
                    'second_event_tuple': second_event_tuple
                })

        return processed_data

    def process_event_tuple(self, event_tuple_str):
        """处理事件五元组字符串，将'无'替换为空字符串"""
        # 使用正则表达式从字符串中提取五元组成分
        pattern = r'\((.*?); (.*?); (.*?); (.*?); (.*?)\)'
        match = re.match(pattern, event_tuple_str)

        if not match:
            return ("", "", "", "", "")

        # 提取五个元素并替换"无"为空字符串
        trigger, subject, object_, time, location = match.groups()
        trigger = "" if trigger == "无" else trigger
        subject = "" if subject == "无" else subject
        object_ = "" if object_ == "无" else object_
        time = "" if time == "无" else time
        location = "" if location == "无" else location

        return (trigger, subject, object_, time, location)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # 获取事件文本和处理过的五元组
        event1_text = item['first_event_text']
        event1_tuple = item['first_event_tuple']
        event2_text = item['second_event_text']
        event2_tuple = item['second_event_tuple']

        # 将五元组转换为字符串，空元素会跳过
        event1_tuple_parts = []
        if event1_tuple[0]:  # 触发词不为空
            event1_tuple_parts.append(f"触发词：{event1_tuple[0]}")
        if event1_tuple[1]:  # 主语不为空
            event1_tuple_parts.append(f"主语：{event1_tuple[1]}")
        if event1_tuple[2]:  # 宾语不为空
            event1_tuple_parts.append(f"宾语：{event1_tuple[2]}")
        if event1_tuple[3]:  # 时间状语不为空
            event1_tuple_parts.append(f"时间状语：{event1_tuple[3]}")
        if event1_tuple[4]:  # 地点状语不为空
            event1_tuple_parts.append(f"地点状语：{event1_tuple[4]}")

        event1_tuple_str = "；".join(event1_tuple_parts) if event1_tuple_parts else ""

        # 同样处理事件2的五元组
        event2_tuple_parts = []
        if event2_tuple[0]:
            event2_tuple_parts.append(f"触发词：{event2_tuple[0]}")
        if event2_tuple[1]:
            event2_tuple_parts.append(f"主语：{event2_tuple[1]}")
        if event2_tuple[2]:
            event2_tuple_parts.append(f"宾语：{event2_tuple[2]}")
        if event2_tuple[3]:
            event2_tuple_parts.append(f"时间状语：{event2_tuple[3]}")
        if event2_tuple[4]:
            event2_tuple_parts.append(f"地点状语：{event2_tuple[4]}")

        event2_tuple_str = "；".join(event2_tuple_parts) if event2_tuple_parts else ""

        # 拼接所有信息，避免空信息
        combined_text = f"事件1原文：{event1_text}"
        if event1_tuple_str:
            combined_text += f" 事件1描述：{event1_tuple_str}"

        combined_text += f" 事件2原文：{event2_text}"
        if event2_tuple_str:
            combined_text += f" 事件2描述：{event2_tuple_str}"

        # 将关系类型映射为数值标签
        relation_mapping = {
            "并列": 0,
            "使能-因果": 1,
            "动机-因果": 2,
            "物理-因果": 3,
            "心理-因果": 4
        }

        label = relation_mapping.get(item['relation_type'], 0)  # 默认为0

        # 使用transformers的tokenizer进行编码
        encoding = self.tokenizer(
            combined_text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # 提取并扁平化张量
        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        token_type_ids = encoding['token_type_ids'].flatten() if 'token_type_ids' in encoding else torch.zeros_like(input_ids)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'token_type_ids': token_type_ids,
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# 定义注意力机制
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.hidden_dim = hidden_dim
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):
        # lstm_output: [batch_size, seq_len, hidden_dim]
        attention_weights = torch.softmax(self.attention(lstm_output).squeeze(-1), dim=1)
        # attention_weights: [batch_size, seq_len]

        # 将注意力权重应用到LSTM输出
        context_vector = torch.bmm(attention_weights.unsqueeze(1), lstm_output)
        # context_vector: [batch_size, 1, hidden_dim]

        return context_vector.squeeze(1), attention_weights

# 定义改进的BiLSTM模型，使用transformers的BERT模型
class EventRelationBiLSTMWithAttention(nn.Module):
    def __init__(self, bert_model, hidden_dim, output_dim, n_layers, dropout):
        super(EventRelationBiLSTMWithAttention, self).__init__()

        # 使用transformers的BERT模型
        self.bert = bert_model

        # BERT输出维度
        self.bert_output_dim = self.bert.config.hidden_size

        # BiLSTM层
        self.lstm = nn.LSTM(
            input_size=self.bert_output_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )

        # 注意力层
        self.attention = Attention(hidden_dim * 2)  # 双向LSTM, 所以hidden_dim*2

        # 输出层
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # 通过BERT获取上下文表示
        if token_type_ids is not None:
            bert_outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        else:
            bert_outputs = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # 获取序列输出 - transformers的BERT模型直接返回last_hidden_state
        sequence_output = bert_outputs.last_hidden_state

        # 通过BiLSTM处理序列
        lstm_output, (hidden, cell) = self.lstm(sequence_output)

        # 应用注意力机制
        context_vector, attention_weights = self.attention(lstm_output)

        # Dropout
        context_vector = self.dropout(context_vector)

        # 全连接层进行分类
        output = self.fc(context_vector)

        return output, attention_weights



In [ ]:
# 训练函数
def train_model(model, data_loader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    epoch_preds = []
    epoch_labels = []

    for batch in data_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)

        # 前向传播
        outputs, _ = model(input_ids, attention_mask, token_type_ids)

        # 计算损失
        loss = criterion(outputs, labels)

        # 反向传播
        loss.backward()

        # 梯度裁剪，防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 更新参数
        optimizer.step()

        epoch_loss += loss.item()

        # 获取预测
        _, preds = torch.max(outputs, 1)
        epoch_preds.extend(preds.cpu().numpy())
        epoch_labels.extend(labels.cpu().numpy())

    # 计算准确率
    accuracy = accuracy_score(epoch_labels, epoch_preds)

    wandb.log({
    "epoch": epoch,
    "train_loss": avg_loss,
    "train_accuracy": accuracy
    })

    return epoch_loss / len(data_loader), accuracy

# 评估函数
def evaluate_model(model, data_loader, criterion, device):
    model.eval()
    epoch_loss = 0
    epoch_preds = []
    epoch_labels = []
    all_attention_weights = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)

            # 前向传播
            outputs, attention_weights = model(input_ids, attention_mask, token_type_ids)

            # 收集注意力权重
            all_attention_weights.append(attention_weights.cpu().numpy())

            # 计算损失
            loss = criterion(outputs, labels)

            epoch_loss += loss.item()

            # 获取预测
            _, preds = torch.max(outputs, 1)
            epoch_preds.extend(preds.cpu().numpy())
            epoch_labels.extend(labels.cpu().numpy())

    # 计算准确率
    accuracy = accuracy_score(epoch_labels, epoch_preds)
    # 生成详细的评估报告
    report = classification_report(epoch_labels, epoch_preds)
    wandb.log({
    "epoch": epoch,
    "val_loss": epoch_loss / len(data_loader),
    "val_accuracy": accuracy
    })

    return epoch_loss / len(data_loader), accuracy, report, np.concatenate(all_attention_weights, axis=0)

# 可视化注意力权重的函数
def visualize_attention(text, attention_weights, tokenizer):
    tokens = tokenizer.tokenize(text)
    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(15, 5))
    sns.heatmap(attention_weights.reshape(1, -1)[:, :len(tokens)], annot=True, cmap='Blues', yticklabels=False)
    plt.xticks(np.arange(len(tokens)), tokens, rotation=90)
    plt.tight_layout()
    plt.title('Attention Weights')
    plt.savefig('attention_weights.png')
    plt.close()

# 创建学习率调度器
def create_scheduler(optimizer, num_warmup_steps, num_training_steps):
    """创建线性学习率调度器"""
    from torch.optim.lr_scheduler import LambdaLR

    def lr_lambda(current_step):
        # 预热阶段
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        # 线性衰减
        return max(
            0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps))
        )

    return LambdaLR(optimizer, lr_lambda)



In [ ]:
import wandb
from google.colab import userdata

wnb_token=userdata.get('wandb_token')
wandb.login(key=wnb_token) # import wandb
run = wandb.init(
    project='BiLSTM_relation_classification-test0416',
    entity='FeSCN',
    job_type="training",
    settings=wandb.Settings(init_timeout=120),
    anonymous="allow"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yuxuan0612 (FeSCN) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
def init_wandb(project_name, config):
    """
    初始化 wandb 项目

    参数:
    project_name: wandb 项目名称
    config: 配置参数字典
    """
    wandb.init(project=project_name, config=config)

    # 可以添加一些额外的配置项
    wandb.run.name = f"bert-bilstm-attn-{wandb.run.id}"
    wandb.run.save()

    return wandb.config

In [ ]:
def main():
    # 设置设备
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")

    # 设置超参数
    # 修改为本地模型路径
    MODEL_NAME = "hfl/chinese-bert-wwm"  # 你的本地BERT模型路径
    HIDDEN_DIM = 256
    OUTPUT_DIM = 5  # 5种关系类别
    N_LAYERS = 2
    DROPOUT = 0.3
    BATCH_SIZE = 32
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    N_EPOCHS = 30

    # 检查模型目录是否存在
    # if not os.path.exists(MODEL_PATH):
        # print(f"警告: 模型目录 {MODEL_PATH} 不存在!")

    # 加载tokenizer和模型
    try:
        tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
        bert_model = BertModel.from_pretrained(MODEL_NAME)
        print("成功加载模型和tokenizer")
    except Exception as e:
        print(f"加载模型失败: {e}")
        raise

    # 准备数据
    data_path_train = "/content/output_relation_train.jsonl"
    data_path_dev = "/content/output_relation_dev.jsonl"

    try:
        # 创建数据集
        train_dataset = EventRelationDataset(data_path_train, tokenizer, max_len=256)
        val_dataset = EventRelationDataset(data_path_dev, tokenizer, max_len=256)

        # 创建数据加载器
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

        print(f"训练集大小: {len(train_dataset)}")
        print(f"验证集大小: {len(val_dataset)}")
    except Exception as e:
        print(f"数据准备失败: {e}")
        raise

    # 初始化模型
    model = EventRelationBiLSTMWithAttention(
        bert_model=bert_model,
        hidden_dim=HIDDEN_DIM,
        output_dim=OUTPUT_DIM,
        n_layers=N_LAYERS,
        dropout=DROPOUT
    ).to(device)

    # 定义优化器，使用不同的学习率
    optimizer_grouped_parameters = [
        {'params': model.bert.parameters(), 'lr': LEARNING_RATE},
        {'params': model.lstm.parameters(), 'lr': LEARNING_RATE * 5},
        {'params': model.attention.parameters(), 'lr': LEARNING_RATE * 5},
        {'params': model.fc.parameters(), 'lr': LEARNING_RATE * 5}
    ]

    optimizer = optim.AdamW(optimizer_grouped_parameters, weight_decay=WEIGHT_DECAY)

    # 定义学习率调度器
    total_steps = len(train_loader) * N_EPOCHS
    warmup_steps = int(0.1 * total_steps)
    scheduler = create_scheduler(optimizer, warmup_steps, total_steps)

    # 定义损失函数
    criterion = nn.CrossEntropyLoss()

    # 训练模型
    best_val_loss = float('inf')

    for epoch in range(N_EPOCHS):
        print(f'Epoch: {epoch+1}/{N_EPOCHS}')

        # 训练
        train_loss, train_acc = train_model(model, train_loader, optimizer, criterion, device)
        print(f'Train Loss: {train_loss:.3f}, Train Acc: {train_acc:.3f}')

        # 学习率调度器步进
        scheduler.step()

        # 验证
        val_loss, val_acc, val_report, attention_weights = evaluate_model(model, val_loader, criterion, device)
        print(f'Val Loss: {val_loss:.3f}, Val Acc: {val_acc:.3f}')
        print(f'Classification Report:\n{val_report}')

        # 可视化一个样本的注意力权重
        '''
        if epoch == N_EPOCHS - 1:  # 在最后一个epoch可视化
            try:
                sample_idx = 0  # 选择一个样本
                sample = val_dataset[sample_idx]
                sample_text = tokenizer.decode(sample['input_ids'])
                sample_attention = attention_weights[sample_idx]
                visualize_attention(sample_text, sample_attention, tokenizer)
                print("注意力权重可视化已保存")
            except Exception as e:
                print(f"可视化失败: {e}")
        '''
        # 保存最佳模型
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            try:
                torch.save(model.state_dict(), 'best_event_relation_model_improved.pt')
                print('模型已保存!')
            except Exception as e:
                print(f"保存模型失败: {e}")

    print('训练完成!')


In [ ]:
main()

使用设备: cuda
成功加载模型和tokenizer
训练集大小: 10502
验证集大小: 2968
Epoch: 1/30


KeyboardInterrupt: 

In [ ]:
wandb.finish()

# 测试集评估

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
def load_model_and_evaluate(model_path, eval_data_path, tokenizer, device, batch_size=32):
    """
    加载保存的模型并在评估数据集上计算各种指标

    参数:
    model_path: 保存的模型路径
    eval_data_path: 评估数据集路径
    tokenizer: 已加载的tokenizer
    device: 计算设备(CPU/GPU)
    batch_size: 批次大小
    """
    # 加载评估数据集
    eval_dataset = EventRelationDataset(eval_data_path, tokenizer, max_len=256)
    eval_loader = DataLoader(eval_dataset, batch_size=batch_size)
    print(f"评估集大小: {len(eval_dataset)}")

    # 初始化与训练时相同结构的模型
    bert_model = BertModel.from_pretrained(MODEL_NAME)
    model = EventRelationBiLSTMWithAttention(
        bert_model=bert_model,
        hidden_dim=256,  # 确保与训练时使用的相同参数
        output_dim=5,    # 确保与训练时相同的类别数
        n_layers=2,
        dropout=0.3
    ).to(device)

    # 加载保存的模型参数
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"成功加载模型参数: {model_path}")

    # 设置为评估模式
    model.eval()

    # 收集预测结果和真实标签
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in eval_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels = batch['labels'].to(device)

            # 前向传播
            outputs, _ = model(input_ids, attention_mask, token_type_ids)

            # 获取预测
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 计算各种指标
    # 1. 准确率
    accuracy = accuracy_score(all_labels, all_preds)

    # 2. 宏观指标 (macro) - 每个类别的指标计算后取平均
    macro_precision = precision_score(all_labels, all_preds, average='macro')
    macro_recall = recall_score(all_labels, all_preds, average='macro')
    macro_f1 = f1_score(all_labels, all_preds, average='macro')

    # 3. 微观指标 (micro) - 将所有类别的样本合并后计算
    micro_precision = precision_score(all_labels, all_preds, average='micro')
    micro_recall = recall_score(all_labels, all_preds, average='micro')
    micro_f1 = f1_score(all_labels, all_preds, average='micro')

    # 4. 加权指标 (weighted) - 考虑类别不平衡的情况
    weighted_precision = precision_score(all_labels, all_preds, average='weighted')
    weighted_recall = recall_score(all_labels, all_preds, average='weighted')
    weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

    # 5. 每个类别的详细指标
    report = classification_report(all_labels, all_preds)

    # 打印结果
    print(f"\n评估结果:")
    print(f"准确率 (Accuracy): {accuracy:.4f}")

    print(f"\n宏观指标 (Macro):")
    print(f"Precision: {macro_precision:.4f}")
    print(f"Recall: {macro_recall:.4f}")
    print(f"F1-score: {macro_f1:.4f}")

    print(f"\n微观指标 (Micro):")
    print(f"Precision: {micro_precision:.4f}")
    print(f"Recall: {micro_recall:.4f}")
    print(f"F1-score: {micro_f1:.4f}")

    print(f"\n加权指标 (Weighted):")
    print(f"Precision: {weighted_precision:.4f}")
    print(f"Recall: {weighted_recall:.4f}")
    print(f"F1-score: {weighted_f1:.4f}")

    print(f"\n各类别详细指标:")
    print(report)

    # 返回指标字典，方便进一步处理
    metrics = {
        "accuracy": accuracy,
        "macro": {
            "precision": macro_precision,
            "recall": macro_recall,
            "f1": macro_f1
        },
        "micro": {
            "precision": micro_precision,
            "recall": micro_recall,
            "f1": micro_f1
        },
        "weighted": {
            "precision": weighted_precision,
            "recall": weighted_recall,
            "f1": weighted_f1
        }
    }

    return metrics, report

In [ ]:
# 最优模型路径和评估数据集路径
best_model_path = 'best_event_relation_model_improved.pt'
eval_data_path = '/content/output_relation_eval.jsonl'  # 替换为您的评估数据集路径

# 加载模型并评估
metrics, report = load_model_and_evaluate(best_model_path, eval_data_path, tokenizer, device)

# 可以选择将结果保存到文件
with open('evaluation_results.json', 'w') as f:
    json.dump(metrics, f, indent=4)